# ROHAN Pipeline Validation

End-to-end validation notebook that exercises every layer of the ROHAN framework:

1. **Configuration** â€” Load simulation settings from a preset
2. **Simulation** â€” Run a baseline simulation via `SimulationService`
3. **Metrics** â€” Compute market & agent metrics via `AnalysisService`
4. **Rich Metrics** â€” Derive L1 time-series and microstructure metrics
5. **Scenario Planning** â€” Run the keyword-fallback planner (no LLM needed)
6. **Scoring** â€” Exercise the deterministic 6-axis scoring pipeline
7. **Persistence** â€” Save and reload a session via `SessionRepository`

This notebook doubles as a regression test and thesis demo.  
It runs fully offline â€” no LLM API keys required.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(name)s] %(levelname)s: %(message)s")
logging.getLogger("abides_markets").setLevel(logging.WARNING)

print("Imports loaded.")

## 1. Configuration

Load a preset simulation configuration and verify the settings object.

In [ ]:
from rohan.ui.utils.presets import get_preset_config, get_preset_names

print("Available presets:")
for name in get_preset_names():
    print(f"  - {name}")

settings = get_preset_config("Default (Balanced Market)")
settings.seed = 42
settings.end_time = "09:35:00"  # 5-minute sim for speed

print(f"\nTemplate: {settings.template}")
print(f"Seed: {settings.seed}")
print(f"Window: {settings.start_time} -> {settings.end_time}")
print("\nConfiguration loaded.")

## 2. Simulation

Run a baseline simulation (no strategy agent) and verify the result.

In [ ]:
from rohan.simulation.simulation_service import SimulationService

service = SimulationService()
sim_result = service.run_simulation(settings)

assert sim_result.error is None, f"Simulation failed: {sim_result.error}"
assert sim_result.result is not None
assert sim_result.duration_seconds > 0

output = sim_result.result
print(f"Simulation completed in {sim_result.duration_seconds:.2f}s")
print(f"Agents: {len(output._result.agents)}")

l1 = output.get_order_book_l1()
assert l1 is not None and len(l1) > 0, "L1 order book is empty"
print(f"L1 snapshots: {len(l1):,}")
print("\nSimulation passed.")

## 3. Metrics Computation

Compute market microstructure metrics and agent-level analytics.

In [ ]:
from rohan.framework.analysis_service import AnalysisService

analyzer = AnalysisService()
metrics = analyzer.compute_metrics(output)

print("Market Metrics:")
print(f"  Mean spread:       {metrics.mean_spread:.2f} cents" if metrics.mean_spread else "  Mean spread:       N/A")
print(f"  Volatility (ann):  {metrics.volatility:.4f}" if metrics.volatility else "  Volatility (ann):  N/A")
print(f"  VPIN:              {metrics.vpin:.4f}" if metrics.vpin else "  VPIN:              N/A")
print(f"  LOB imbalance:     {metrics.lob_imbalance_mean:.4f}" if metrics.lob_imbalance_mean else "  LOB imbalance:     N/A")
print(f"  Traded volume:     {metrics.traded_volume:,}" if metrics.traded_volume else "  Traded volume:     N/A")

assert metrics.mean_spread is not None, "Mean spread should be computable"
print("\nMetrics computed.")

## 4. Rich Metrics (L1-Derived)

Access hasufel's native L1 snapshots and derive spread/return time-series.

In [ ]:
from rohan.ui.metrics import compute_microstructure_metrics, compute_spread_stats, derive_l1

hasufel_result = output._result  # Access the underlying hasufel SimulationResult

# Get L1 data via hasufel's native path (columns: time_ns, bid_price_cents, ask_price_cents)
l1_hasufel = hasufel_result.markets[settings.ticker].l1_series.as_dataframe()
print(f"Hasufel L1 columns: {list(l1_hasufel.columns)}")
print(f"Hasufel L1 rows: {len(l1_hasufel):,}")

# L1-derived metrics
l1_derived = derive_l1(l1_hasufel)
spread_stats = compute_spread_stats(l1_derived.spread, l1_derived.mid)
print(f"Spread stats: mean={spread_stats.mean:.4f}, median={spread_stats.median:.4f}, two-sided={spread_stats.n_two_sided}/{spread_stats.n_total}")

# Microstructure metrics (uses hasufel's SimulationResult directly)
micro = compute_microstructure_metrics(hasufel_result, settings.ticker)
if micro:
    print(f"Microstructure: spread={micro.mean_spread_cents:.2f}c" if micro.mean_spread_cents else "Microstructure: spread=N/A")
    print(f"  Volatility (ann): {micro.volatility_ann:.4f}" if micro.volatility_ann else "  Volatility: N/A")
    print(f"  Two-sided time: {micro.pct_time_two_sided:.1f}%" if micro.pct_time_two_sided else "  Two-sided: N/A")
else:
    print("Microstructure metrics unavailable (too short simulation?)")

print("\nRich metrics computed.")

## 5. Scenario Planning

Run the keyword-fallback planner (tier 3) to verify scenario selection logic without an LLM.

In [ ]:
from unittest.mock import MagicMock

from rohan.llm.planner import _keyword_fallback, plan_scenarios
from rohan.llm.state import ScenarioConfig

# Direct keyword fallback test
goal = "Create a market-making strategy that profits from the bid-ask spread"
adversarial = _keyword_fallback(goal, max_adversarial=2)
print(f"Keyword fallback for '{goal[:50]}...'")
for s in adversarial:
    print(f"  -> {s.name} (template={s.template_name}, tags={s.regime_tags})")
    print(f"    Rationale: {s.rationale}")

# Full planner with max_adversarial=0 (disabled -- no LLM needed)
user_scenarios = [ScenarioConfig(name="baseline")]

mock_settings = MagicMock()
mock_settings.max_adversarial_scenarios = 0
scenarios, reasoning = plan_scenarios(goal, user_scenarios, mock_settings)
assert len(scenarios) == 1
assert scenarios[0].name == "baseline"
print(f"\nDisabled planner: {len(scenarios)} scenario(s) -- {reasoning}")

print("\nScenario planning validated.")

## 6. Deterministic Scoring

Exercise the 6-axis scoring pipeline with synthetic metrics to verify formula correctness.

In [ ]:
from rohan.llm.scoring import WEIGHT_PROFILES, compute_axis_scores, compute_final_score

# Compute axis scores using the deterministic scoring API
axes = compute_axis_scores(
    strategy_pnl=500.0,
    trade_count=150,
    sharpe_ratio=1.2,
    max_drawdown=-200.0,
    fill_rate=85.0,
    order_to_trade_ratio=3.5,
    volatility_delta_pct=-2.0,
    spread_delta_pct=1.5,
    bid_liquidity_delta_pct=5.0,
    ask_liquidity_delta_pct=-3.0,
    starting_capital_cents=10_000_00,  # $10,000
    baseline_mean_spread=8.60,
    baseline_traded_volume=85_000.0,
    pct_time_two_sided_delta=-1.0,
    avg_slippage_bps=2.5,
)

weights = WEIGHT_PROFILES["default"]
final = compute_final_score(axes, weights)

print("6-Axis Scores:")
print(f"  Profitability:    {axes.profitability:.2f}/10")
print(f"  Risk-Adjusted:    {axes.risk_adjusted:.2f}/10")
print(f"  Vol Impact:       {axes.volatility_impact:.2f}/10")
print(f"  Spread Impact:    {axes.spread_impact:.2f}/10")
print(f"  Liquidity Impact: {axes.liquidity_impact:.2f}/10")
print(f"  Execution:        {axes.execution_quality:.2f}/10")
print(f"\n  Final Score:      {final:.2f}/10")

for field_name in ["profitability", "risk_adjusted", "volatility_impact", "spread_impact", "liquidity_impact", "execution_quality"]:
    val = getattr(axes, field_name)
    assert 1.0 <= val <= 10.0, f"{field_name}={val} out of range"
assert 1.0 <= final <= 10.0

print("\nScoring validated -- all axes in [1, 10].")

## 7. Persistence

Save and reload a mock session via `SessionRepository` to validate the persistence layer.

In [ ]:
# Use in-memory SQLite for validation
import os

from rohan.framework.database import get_database_connector, initialize_database
from rohan.framework.repository import IterationData, ScenarioRunData, SessionRepository

os.environ["ROHAN_DATABASE_URL"] = "sqlite:///:memory:"
get_database_connector.cache_clear()
initialize_database()

repo = SessionRepository()

session = repo.save_session(
    name="pipeline_validation_test",
    goal=goal,
    max_iterations=3,
    scenario_configs=[{"name": "baseline"}, {"name": "volatile_stress"}],
    status="completed",
    final_score=7.5,
    total_duration=42.0,
    progress_log=["Iteration 1 done", "Iteration 2 done"],
    final_code="class TestStrategy:\n    pass",
    final_class_name="TestStrategy",
    final_reasoning="Converged after 2 iterations",
    iterations=[
        IterationData(
            iteration_number=1,
            strategy_code="class TestStrategy:\n    pass",
            class_name="TestStrategy",
            scenario_runs=[
                ScenarioRunData(scenario_name="baseline", domain_metrics={"pnl": 100.0}),
            ],
        ),
    ],
)
print(f"Session saved: id={session.session_id}, name={session.name}")

# Reload
loaded = repo.load_session(session.session_id)
assert loaded is not None, "Failed to load saved session"
assert loaded["refine_goal"] == goal
assert loaded["refine_final_state"]["status"] == "completed"
print(f"Session reloaded: goal='{loaded['refine_goal'][:50]}...'")

# List sessions
summaries = repo.list_sessions()
assert len(summaries) >= 1
print(f"Sessions in DB: {len(summaries)}")
print(f"  -> {summaries[0].name} (score={summaries[0].final_score})")

print("\nPersistence validated.")

## Summary

All pipeline stages validated:

| Stage | Status |
|-------|--------|
| Configuration (preset loading) | Pass |
| Simulation (baseline execution) | Pass |
| Metrics (market + microstructure) | Pass |
| Rich Metrics (L1-derived) | Pass |
| Scenario Planning (keyword fallback) | Pass |
| Scoring (6-axis deterministic) | Pass |
| Persistence (save/load session) | Pass |

The full pipeline works end-to-end without LLM API keys.